#Processign Silver layer for account_move_line model

For create the silver account_move_line I get temp view for load all data from the bronze layer, and complete the 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_account_move_line AS
SELECT 
id AS account_move_line_id,
move_id[0] AS move_id,
move_id[1] AS move_name,
account_id[0] AS account_id,
account_id[1] AS account_name,
amount_currency AS amount_currency,
amount_residual AS amount_residual,
balance AS balance,
credit AS credit,
debit AS debit,
CAST(date AS DATE) AS account_move_date,
name AS name,
CAST(CASE
        WHEN payment_id IS NULL OR payment_id = 'false' THEN NULL
        ELSE get_json_object(payment_id, '$[0]')
        END AS INT
    ) AS payment_id,
CAST(CASE
        WHEN payment_id IS NULL OR payment_id = 'false' THEN NULL
        ELSE get_json_object(payment_id, '$[1]')
        END AS STRING
    ) AS payment_move,
price_subtotal AS price_subtotal,
price_total AS price_total, 
price_unit AS price_unit,
CAST (CASE
        WHEN product_id IS NULL OR product_id = 'false' THEN NULL
        ELSE get_json_object(product_id, '$[0]')
        END AS INT
    ) AS product_id,
CAST (CASE
        WHEN product_id IS NULL OR product_id = 'false' THEN NULL
        ELSE get_json_object(product_id, '$[1]')
        END AS STRING
    ) AS product_name,
quantity AS quantity,
ref AS ref,
tax_base_amount AS tax_base_amount,
CAST(write_date AS TIMESTAMP) AS updated_at,
write_uid[0] AS updated_by,
processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.account_move_line
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id 
    ORDER BY processing_timestamp DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.account_move_line (
    account_move_line_id INT,
    move_id INT,
    move_name STRING,
    account_id INT,
    account_name STRING,
    amount_currency DECIMAL(10,2),
    amount_residual DECIMAL(10, 2),
    balance FLOAT,
    credit FLOAT,
    debit FLOAT,
    account_move_date DATE,
    name STRING,
    payment_id INT,
    payment_move STRING,
    price_subtotal DECIMAL(10,2),
    price_total DECIMAL(10,2),
    price_unit DECIMAL(10,2),
    product_id INT,
    product_name STRING,
    quantity FLOAT,
    ref STRING,
    tax_base_amount DECIMAL(10,2),
    updated_at TIMESTAMP,
    updated_by INT,
    bronze_ingested_at TIMESTAMP,
    state STRING,
    silver_processed_at TIMESTAMP
)

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move_line AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_account_move_line
) AS source
ON target.account_move_line_id = source.account_move_line_id
WHEN MATCHED AND target.updated_at < source.updated_at THEN
  UPDATE SET
    move_id = source.move_id,
    move_name = source.move_name,
    account_id = source.account_id,
    account_name = source.account_name,
    amount_currency = source.amount_currency,
    amount_residual = source.amount_residual,
    balance = source.balance,
    credit = source.credit,
    debit = source.debit,
    account_move_date = source.account_move_date,
    name = source.name,
    payment_id = source.payment_id,
    payment_move = source.payment_move,
    price_subtotal = source.price_subtotal,
    price_total = source.price_total,
    price_unit = source.price_unit,
    product_id = source.product_id,
    product_name = source.product_name,
    quantity = source.quantity,
    ref = source.ref,
    tax_base_amount = source.tax_base_amount,
    updated_at = source.updated_at,
    updated_by = source.updated_by,
    bronze_ingested_at = source.bronze_ingested_at,
    state = 'updated',
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN 
  INSERT (
    account_move_line_id, move_id, move_name, account_id, account_name, 
    amount_currency, amount_residual, balance, credit, debit, 
    account_move_date, name, payment_id, payment_move, price_subtotal, 
    price_total, price_unit, product_id, product_name, quantity, 
    ref, tax_base_amount, updated_at, updated_by, bronze_ingested_at, 
    state, silver_processed_at
  )
  VALUES (
    source.account_move_line_id, source.move_id, source.move_name, source.account_id, source.account_name, 
    source.amount_currency, source.amount_residual, source.balance, source.credit, source.debit, 
    source.account_move_date, source.name, source.payment_id, source.payment_move, source.price_subtotal, 
    source.price_total, source.price_unit, source.product_id, source.product_name, source.quantity, 
    source.ref, source.tax_base_amount, source.updated_at, source.updated_by, source.bronze_ingested_at, 
    'inserted', source.silver_processed_at
  );


In [0]:
%sql
DROP VIEW IF EXISTS stg_delete_account_move

In [0]:
%sql

CREATE TEMP VIEW stg_delete_account_move AS
SELECT * FROM bakery_pos_develop.silver.account_move WHERE state = 'deleted'


In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.account_move_line AS target
USING stg_delete_account_move AS source
ON target.move_id = source.account_move_id
WHEN MATCHED THEN 
  UPDATE SET
    updated_by = source.updated_by,
    state = 'deleted',
    silver_processed_at = source.silver_processed_at;
